In [ ]:
import sys
import os

# Add the repository root to sys.path
sys.path.append(os.path.abspath(".."))

import numpy as np
import nibabel as nib 
import os 
import pandas as pd
from config import load_config, Configuration

config = load_config("config.yaml")
mask_dir = os.path.join(config.stans_thesis_repo_data, config.mask_data_dir)

In [ ]:
columns = ["data"]
hemis = ['lh', 'rh']
fields = columns
subs = [f"subj{i:02d}" for i in range(1, 8+1)]


mode = "signed"
assert mode in ["absolute", "signed"]
thresholding = 5
            
s = -1


for sub_i, sub in enumerate(subs):
    print(f"Sub: {sub_i}")

    t_data = np.load(os.path.join(config.t_test_results_dir, f"result_subj_{(sub_i+1):02d}.npy"))  
    print(f"t_data_shape: {t_data.shape}")

    max_value = np.nanmax(t_data[:, 0]).item()
    min_value = np.nanmin(t_data[:, 0]).item()

    hemisphere_shapes = []
    

    for hemi_i, hemi in enumerate(hemis):        


        if sub == 'subj06' or sub == 'subj08':
            maskdata_long_file = os.path.join(mask_dir, sub, f'{hemi}.{sub}.nans.testrois.mgz')
        else:
            maskdata_long_file = os.path.join(mask_dir, sub, f'{hemi}.{sub}.testrois.mgz')
        maskdata_long = nib.load(maskdata_long_file).get_fdata().squeeze()
        # print(f"{sub=}\t{hemi=}\t{maskdata_long.shape=}")
        print(f"maskdata: {maskdata_long.shape}")


        hemisphere_shapes.append(maskdata_long.shape[0])

        maskdata_file = os.path.join(mask_dir, sub, f'{hemi}.short.reduced.{sub}.testrois.npy')
        maskdata = np.load(maskdata_file, allow_pickle=True).astype(int)
        # print(f"{sub=}\t{hemi=}\t{maskdata_long.shape=}")

        

        # data_out_file = os.path.join(config.t_test_results_dir, "matlab", f'{sub}.{hemi}.mgz')
        data_out_file = os.path.join(config.freesurfer_dir, sub, "label", f"{hemi}.t_test_{mode}.mgz")
        #label_out_file = os.path.join(label_dir,'freesurfer', sub, 'label', f'{hemi}.{model}.{f}.{rotation}.mgz') # annoying but necessary for matlab output
        # if os.path.exists(data_out_file):
        #     pass
        data_out = np.zeros(maskdata_long.shape)
        
        wrongs = []
        for i in range(maskdata_long.shape[0]):
            #voxel = int(maskdata_long[i])
            #if voxel==0 or voxel > 15:

            #    continue
            #s += 1
            
            if hemi_i == 0:
                index = i
            else:
                index = i + hemisphere_shapes[0]

            if t_data[index][0] < thresholding:
                continue
            
            

            if mode == "absolute":
                data_out[i] = np.abs(t_data[index][0])
            else:
                data_out[i] = t_data[index][0] + np.abs(min_value)
        
        print(f"data_out_shape: {data_out.shape}")

        


        img = nib.Nifti1Image(np.expand_dims(data_out, axis=(1, 2)), np.eye(4))
        nib.loadsave.save(img, data_out_file)

    if mode == "signed":
        range_max = max(max_value, np.abs(min_value))
        val_range = [-range_max, range_max]
        adjusted_range = [0, 2*range_max]


        print(f"Data range: {val_range}")
        print(f"Adjusted Data range: {adjusted_range}")
        print(f"Maximal Value: {max_value}")
        print(f"Minimal Value: {min_value}")

    else:
        print(f"Maximal Value: {max_value}")
        print(f"Minimal Value: {min_value}")

    print("-"*50)
    

        



